# NB08 · Beginner Capstone

`beginner/` · one story project in 6 milestones

The beginner finale. You'll build a complete little program, a **bank
simulator**, in six milestones: formatting money, deposits and withdrawals,
transfers, processing a whole day of transactions, printing account
statements, and a final report. Every milestone uses the functions you wrote
in the ones before, and everything from NB00 to NB07 is fair game.

## You'll need

- NB00 to NB07: all beginner notebooks

## You'll practise

- Building one bigger program (about 100 to 150 lines) step by step
- Choosing which beginner tools to use for each part
- Splitting a program into small functions that work together

## How it works

Write your code in the cell under each question, run it, and compare with **Expected output**. Open **Solution** only after a real attempt.

Each milestone builds on the one before, so run the cells in order.

---

## Byte Bank

You've been hired as the first developer at *Byte Bank*, a tiny bank with
three customers. Each night a file of the day's requests arrives, and so far
a clerk has processed it by hand. Some requests are valid, some are not:
overdrafts, unknown customers, nonsense amounts. Build the program that
processes the file and prints each customer's statement. Run the starting
data first, then work through the milestones in order.

**Run these cells first.** They set up the starting files and data for every step below.

In [ ]:
accounts = {"Ana": 500.0, "Bo": 120.0, "Cy": 0.0}

# (kind, account, amount as text)  or  ("transfer", from, to, amount as text)
transactions = [
    ("deposit", "Ana", "200"),
    ("withdraw", "Bo", "150"),
    ("transfer", "Ana", "Cy", "75.50"),
    ("withdraw", "Cy", "abc"),
    ("deposit", "Dee", "50"),
    ("transfer", "Bo", "Ana", "20"),
    ("deposit", "Bo", "-5"),
    ("withdraw", "Ana", "300"),
]


## Milestone 1

Write `money(amount)` that returns a number as text with a dollar sign, a thousands separator and 2 decimals. Then write `show_balances(accounts)` that prints one line per customer in alphabetical order, with the name padded to 6 characters and the money right-aligned in 12 characters. Call it on the starting accounts.

**Expected output:**

```text
Ana        $500.00
Bo         $120.00
Cy           $0.00
```

<details><summary><b>Solution</b></summary>

```python
def money(amount):
    return f"${amount:,.2f}"

def show_balances(accounts):
    for name in sorted(accounts):
        print(f"{name:<6}{money(accounts[name]):>12}")

show_balances(accounts)
```

</details>

In [ ]:
# your code here


## Milestone 2

Write `deposit(accounts, name, amount)` and `withdraw(accounts, name, amount)`. Each changes the balance and returns `True`, or changes nothing and returns `False` when the customer doesn't exist, the amount isn't positive, or (for a withdrawal) the balance is too small. Test on a separate dict `trial = {"Ana": 100.0, "Bo": 50.0}`: deposit 25 to Ana, withdraw 80 from Bo, withdraw 30 from Bo, and deposit 10 to "Zed". Print each result, then the trial balances with `show_balances`.

**Expected output:**

```text
True
False
True
False
Ana        $125.00
Bo          $20.00
```

<details><summary><b>Solution</b></summary>

```python
def deposit(accounts, name, amount):
    if name not in accounts or amount <= 0:
        return False
    accounts[name] += amount
    return True

def withdraw(accounts, name, amount):
    if name not in accounts or amount <= 0 or amount > accounts[name]:
        return False
    accounts[name] -= amount
    return True

trial = {"Ana": 100.0, "Bo": 50.0}
print(deposit(trial, "Ana", 25))
print(withdraw(trial, "Bo", 80))
print(withdraw(trial, "Bo", 30))
print(deposit(trial, "Zed", 10))
show_balances(trial)
```

</details>

In [ ]:
# your code here


## Milestone 3

Write `transfer(accounts, source, target, amount)` using your two functions. It must check that the target exists first, then withdraw from the source, and only if that worked deposit into the target. It returns `True` or `False` like the others, and money must never vanish on a failed transfer. Test it on `trial` with a valid transfer of 40 from Ana to Bo, one of 500 from Bo to Ana, and one of 5 from Ana to "Zed". Print each result and the balances.

**Expected output:**

```text
True
False
False
Ana         $85.00
Bo          $60.00
```

<details><summary><b>Solution</b></summary>

```python
def transfer(accounts, source, target, amount):
    if target not in accounts:
        return False
    if withdraw(accounts, source, amount):
        deposit(accounts, target, amount)
        return True
    return False

print(transfer(trial, "Ana", "Bo", 40))
print(transfer(trial, "Bo", "Ana", 500))
print(transfer(trial, "Ana", "Zed", 5))
show_balances(trial)
```

</details>

In [ ]:
# your code here


## Milestone 4

Write `process(accounts, transactions)`. For each transaction (numbered from 1) convert the amount text with `float()`; if that raises `ValueError`, the status is `"bad amount"`. Otherwise use `if`/`elif` on the kind to call your functions, giving status `"ok"` or `"rejected"`. Append a tuple `(number, kind, name, other, amount, status)` to a log list, where `other` is the target for transfers and an empty string otherwise (use 0.0 for a bad amount), and return the log. Save `opening = accounts.copy()`, run `process` on the real accounts, then print one line per log entry showing its number, kind, name and status, and the final balances.

**Expected output:**

```text
#1 deposit Ana: ok
#2 withdraw Bo: rejected
#3 transfer Ana: ok
#4 withdraw Cy: bad amount
#5 deposit Dee: rejected
#6 transfer Bo: ok
#7 deposit Bo: rejected
#8 withdraw Ana: ok
Ana        $344.50
Bo         $100.00
Cy          $75.50
```

<details><summary><b>Solution</b></summary>

```python
def process(accounts, transactions):
    log = []
    number = 0
    for tx in transactions:
        number += 1
        kind = tx[0]
        name = tx[1]
        other = ""
        text = tx[2]
        if kind == "transfer":
            other = tx[2]
            text = tx[3]
        try:
            amount = float(text)
        except ValueError:
            log.append((number, kind, name, other, 0.0, "bad amount"))
            continue
        if kind == "deposit":
            done = deposit(accounts, name, amount)
        elif kind == "withdraw":
            done = withdraw(accounts, name, amount)
        else:
            done = transfer(accounts, name, other, amount)
        if done:
            status = "ok"
        else:
            status = "rejected"
        log.append((number, kind, name, other, amount, status))
    return log

opening = accounts.copy()
log = process(accounts, transactions)
for entry in log:
    print(f"#{entry[0]} {entry[1]} {entry[2]}: {entry[5]}")
show_balances(accounts)
```

</details>

In [ ]:
# your code here


## Milestone 5

Write `statement(name, opening, log)` that prints a statement for one customer. First a header line with the name and opening balance. Then, for each log entry with status `"ok"` that involves the customer, one line with the transaction number, the kind, the signed amount (`+` when money comes in, `-` when it goes out, 2 decimals) and the running balance. A transfer counts as out for the sender and in for the receiver. Print the statement for "Ana".

**Expected output:**

```text
--- Ana (opening $500.00) ---
#1 deposit    +200.00    700.00
#3 transfer    -75.50    624.50
#6 transfer    +20.00    644.50
#8 withdraw   -300.00    344.50
```

<details><summary><b>Solution</b></summary>

```python
def statement(name, opening, log):
    balance = opening[name]
    print(f"--- {name} (opening {money(balance)}) ---")
    for number, kind, who, other, amount, status in log:
        if status != "ok":
            continue
        change = 0.0
        if kind == "deposit" and who == name:
            change = amount
        elif kind == "withdraw" and who == name:
            change = -amount
        elif kind == "transfer" and who == name:
            change = -amount
        elif kind == "transfer" and other == name:
            change = amount
        if change != 0.0:
            balance += change
            print(f"#{number} {kind:<9}{change:>+9.2f}{balance:>10.2f}")

statement("Ana", opening, log)
```

</details>

In [ ]:
# your code here


## Milestone 6

Time for the finale. Print a title line, then the statement of every customer in alphabetical order. After that print how many transactions were rejected or had a bad amount, followed by a list of their numbers. Finally print the total money held by the bank using `money`, and the name of the richest customer. Everything uses the data and functions you've already built.

**Expected output:**

```text
=== BYTE BANK: END OF DAY ===
--- Ana (opening $500.00) ---
#1 deposit    +200.00    700.00
#3 transfer    -75.50    624.50
#6 transfer    +20.00    644.50
#8 withdraw   -300.00    344.50
--- Bo (opening $120.00) ---
#6 transfer    -20.00    100.00
--- Cy (opening $0.00) ---
#3 transfer    +75.50     75.50
Failed transactions: 4 [2, 4, 5, 7]
Total held: $520.00
Richest customer: Ana
```

<details><summary><b>Solution</b></summary>

```python
print("=== BYTE BANK: END OF DAY ===")
for name in sorted(opening):
    statement(name, opening, log)

failed = []
for entry in log:
    if entry[5] != "ok":
        failed.append(entry[0])
print("Failed transactions:", len(failed), failed)

total = 0.0
richest = ""
best = -1.0
for name in accounts:
    total += accounts[name]
    if accounts[name] > best:
        best = accounts[name]
        richest = name
print("Total held:", money(total))
print("Richest customer:", richest)
```

</details>

In [ ]:
# your code here


---

## Done

Next up: **NB09 · Functions Advanced** in `intermediate/`.